[Back to Content](../../../content.md)

# Queue
The queue data structure is a linear dynamic set of entities in a specific sequence. It can be modified by adding entities, at one end of the structure, and removing them from the other. Thus, a queue implements a **FIFO** policy (i.e., _first-in, first-out_). In other words, the first element added to the queue will be the first element to be removed.

In the queue, elements are added at the end of the sequence, and removed at the beginning. By convention, the end is also known as *back*, *rear*, or ***tail*** (we will stick to this term), and the beginning is known as *front* or ***head*** (we will stick to this term).

The queue has three main operations: insertion (i.e., enqueue), deletion (i.e., dequeue), and peek.

Unlike other structures, such as arrays or linked lists, the queue cannot be accessed via index (not even to search if the queue has a value). You would have to delete and get the element as many times as you need. However, this can be synthetically done with an auxiliary buffer to avoid modifications.

![Queue](assets/queue.png)

## Queue Nodes

The following is the implementation of the Queue Node.


In [1]:
class QueueNode {
    constructor(value, next = null) {
        this.value = value
        this.next = next
    }
}

While you can perform certain operations using this data structure, you must be careful because accessing the queue elements is done via referencing the head node. As a result, the head accessor can be lost, or if many objects could reference a node that was supposed to be the head, it has changed.

In [2]:
head = new QueueNode(1, new QueueNode(2, new QueueNode(3)));
console.log("Head:\n", head);
while(head){
    head = head.next;
}
console.log("Head:\n", head);

Head:
 QueueNode {
  value: 1,
  next: QueueNode { value: 2, next: QueueNode { value: 3, next: null } }
}
Head:
 null


To solve this nuance we can use a Queue wrapper that keeps track of the head and tail adding and removing elements in and out of the set.

## The Queue (Wrapper)
Let's use a basic implementation of the Queue wrapper to see how using it avoids losing track of the head unless we actually delete an element of the queue.

**Note:** I'll be changing the `Queue` implementation to explain better in a concise manner each operation. The full implementation of the [`QueueNode`](./QueueNode.js), the [`Queue`](./Queue.js), the [`QueueNode` and `Queue` tests](./__test__/queue.spec.js) can be checked for further analysis.

In [3]:
class BasicQueue {
    constructor() {
        this.head = null;
        this.tail = null;
    }
    
    enqueue(value) {
        const node = new QueueNode(value);
    
        if (!this.head) {
            this.head = node;
            this.tail = node;

            return this;
        }
    
        this.tail.next = node;
        this.tail = node;

        return this;
    }

    toString() {
        const nodes = [];
        let currentNode = this.head;
        while (currentNode) {
            nodes.push(currentNode.value);
            currentNode = currentNode.next;
        }
        nodes.push('null');
        return nodes.join('=>');
    }
}


Let's test the previous case using this basic queue wrapper.

In [4]:
queue = new BasicQueue()
queue
    .enqueue(1)
    .enqueue(2)
    .enqueue(3)
    .enqueue(4)
    .enqueue(5)

console.log("Queue:\n", queue.toString(), "\n");
head = queue.head;
console.log("Head:\n", head);
while(head){
    head = head.next;
}
console.log("=== After losing head's track ===")
console.log("Head:\n", head);
console.log("Queue:\n", queue.toString(), "\n");

Queue:
 1=>2=>3=>4=>5=>null 

Head:
 QueueNode {
  value: 1,
  next: QueueNode {
    value: 2,
    next: QueueNode { value: 3, next: [QueueNode] }
  }
}
=== After losing head's track ===
Head:
 null
Queue:
 1=>2=>3=>4=>5=>null 



Again, despite having lost track of the nodes in the `head` variable, the queue is still intact because the wrapper has the track of the head.

## Operations
### `queue.enqueue`
![Enqueue](./enqueue.png#normal)

**Enqueue** is the insertion operation. The insertion occurs at the end of the queue. When the queue is empty the insertion adds a queue node and the wrapper tracks it in the head. As the node is the only element in the queue it has been tracked in the *head* but in the *tail* as well.

However, as more elements are enqueued to the set, the tail keeps pointing to the tail.

Enqueueing has $O(1)$ time complexity because the node is always added at the end of the queue.

### `queue.dequeue`
![Dequeue](./assets/dequeue.png)

**Dequeue** is the deletion operation. The deletion occurs at the beginning of the queue. When the queue is empty the deletion is not performed. On the other hand, when the queue has elements the *head* reference goes to its current *next*, and the *head* is returned to the caller. This has no impact on the *tail* reference.

Dequeueing has $O(1)$ time complexity too because the *head* reference is moved to the next node.

In [5]:
class QueueWithEnqueueAndDequeue {
    constructor() {
        this.head = null;
        this.tail = null;
    }
    
    enqueue(value) {
        const node = new QueueNode(value);
    
        if (!this.head) {
            this.head = node;
            this.tail = node;

            return this;
        }
    
        this.tail.next = node;
        this.tail = node;

        return this;
    }

    dequeue() {
        if (!this.head) return null;
    
        let deletedNode = this.head;
        if (this.head === this.tail) this.tail = null;
        this.head = this.head.next;
        this.size--;
    
        return deletedNode;
    }

    toString() {
        const nodes = [];
        let currentNode = this.head;
        while (currentNode) {
            nodes.push(currentNode.value);
            currentNode = currentNode.next;
        }
        nodes.push('null');
        return nodes.join('=>');
    }
}

In [6]:
queue = new QueueWithEnqueueAndDequeue()
queue
    .enqueue(1)
    .enqueue(2)
    .enqueue(3)
    .enqueue(4)
    .enqueue(5)


console.log("Queue:\n", queue.toString(), "\n");
queue.dequeue()
console.log("Queue:\n", queue.toString(), "\n");

Queue:
 1=>2=>3=>4=>5=>null 

Queue:
 2=>3=>4=>5=>null 



### `queue.peek`
The `peek` function looks at the head of the queue without popping it out from the queue. With this operation, we always have a way to look at the head value. No matter how many elements are enqueued, this operation will always look to the *head*.

In [7]:
class QueueWithPeek {
    constructor() {
        this.head = null;
        this.tail = null;
    }
    
    enqueue(value) {
        const node = new QueueNode(value);
    
        if (!this.head) {
            this.head = node;
            this.tail = node;

            return this;
        }
    
        this.tail.next = node;
        this.tail = node;

        return this;
    }

    peek() {
        if (!this.head) return null;
    
        return this.head.value;
    }
}

In [8]:
queue = new QueueWithPeek()
queue
    .enqueue(1)
    .enqueue(2)
    .enqueue(3)
    .enqueue(4)
    .enqueue(5)

console.log("Peeking: ", queue.peek())

Peeking:  1


## Runtime Complexity Overview
| Operation | Runtime Complexity |
|:---------:|:-------------------:|
| Enqueue   | $O(1)$              |
| Dequeue   | $O(1)$              |
| Peek      | $O(1)$              |

## Space Complexity
Linked lists have $O(n)$ space complexity.

## Bibliography
1. JavaScript Data Structures and Algorithms[^1]
2. Data Structures and Algorithm Analysis in Java[^2]
3. Introduction to Algorithms[^3]
4. Queue (abstract data type)[^4]
 
[^1]: https://doi.org/10.1007/978-1-4842-3988-9
[^2]: https://www.pearson.com/en-us/subject-catalog/p/data-structures-and-algorithm-analysis-in-java/P200000003475/9780137518821 
[^3]: https://archive.org/details/introduction-to-algorithms-third-edition-2009
[^4]: https://en.wikipedia.org/wiki/Queue_(abstract_data_type)

[Back to Content](../../../content.md)